In [14]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)
%cd /content

# 切到你的專案資料夾
%cd /content/drive/MyDrive/LSTM_PROGRAM

##################################
# GARCH-X，照計畫書的公式，但效果很差， VAR的 STD 是用 GARCH的SHAPE

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content
/content/drive/MyDrive/LSTM_PROGRAM


In [15]:
!mkdir -p ~/.ssh
!cp /content/drive/MyDrive/.ssh/id_ed25519* ~/.ssh/
!chmod 700 ~/.ssh
!chmod 600 ~/.ssh/id_ed25519

!eval "$(ssh-agent -s)" && ssh-add ~/.ssh/id_ed25519
!ssh-keyscan github.com >> ~/.ssh/known_hosts
!chmod 644 ~/.ssh/known_hosts

!ssh -T git@github.com

!git config --global user.email "joemi7878@gmail.com"
!git config --global user.name "joemi78"

Agent pid 21568
Identity added: /root/.ssh/id_ed25519 (joemi7878@gmail.com)
# github.com:22 SSH-2.0-2b2e4ed
# github.com:22 SSH-2.0-2b2e4ed
# github.com:22 SSH-2.0-2b2e4ed
# github.com:22 SSH-2.0-2b2e4ed
# github.com:22 SSH-2.0-2b2e4ed
Hi joemi78! You've successfully authenticated, but GitHub does not provide shell access.


In [17]:
!pip install arch

!sudo apt-get update
!sudo apt-get install -y build-essential python3-dev r-base-dev

!pip install -U jedi
!pip install -U pip setuptools wheel

Hit:1 https://cli.github.com/packages stable InRelease
Hit:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:3 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:4 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:5 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Hit:6 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:10 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 Packages [4,170 kB]
Get:11 http://archive.ubuntu.com/ubuntu jammy-updates/universe amd64 Packages [1,615 kB]
Fetched 6,043 kB in 2s (3,271 kB/s)
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provi

In [18]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import r2_score
import os

import tensorflow as tf
from tensorflow.keras import layers, models, callbacks, optimizers
from tensorflow.keras import mixed_precision

from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.metrics import precision_score, recall_score, f1_score

from functools import cached_property
from typing import Union, Optional, Dict

from scipy.stats import norm, t as tdist
from scipy.stats import chi2


try:
    from arch import arch_model
    HAS_ARCH = True
except Exception:
    HAS_ARCH = False


#### SET GPU
gpus = tf.config.list_physical_devices('GPU')
print("Num GPUs:", len(gpus), gpus)

if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print("✓ memory growth set")
    except RuntimeError as e:
        print("⚠️ GPU 已初始化，無法再設定 memory growth：", e)


# （建議）再開啟 XLA 與混合精度
tf.config.optimizer.set_jit(True)
from tensorflow.keras import mixed_precision
mixed_precision.set_global_policy('mixed_float16')

print("是否可用GPU:", tf.test.is_gpu_available())
print("使用中的裝置:", tf.config.list_physical_devices('GPU'))

print("✓ GPU 初始化流程完成")


# 1. 定義檔案路徑
file_paths = {
    # "bonds_day": "./filtered_output/bonds_day_clean_period.csv",
    # "bonds_hour": "./filtered_output/bonds_hour_clean_period.csv",
    # "crypto_day": "./filtered_output/crypto_day_clean_period.csv",
    # "crypto_hour": "./filtered_output/crypto_hour_clean_period.csv",
    # "others_day": "./filtered_output/others_day_clean_period.csv",
    # "others_hour": "./filtered_output/others_hour_clean_period.csv",
    "stock_day":  "./filtered_output/stock_day_fluctuation_aligned.csv"
    # "stock_hour": "./filtered_output/stock_hour_clean_period.csv"
}


def find_date_col(df):
    for col in df.columns:
        if 'date' in col.lower():
            return col
    return df.columns[0]

def read_and_clean(file):
    # ① 正确地读 CSV，不要写 (index=True)
    df = pd.read_csv(file)

    # ② 找到原始时间列名
    date_col = find_date_col(df)

    # ③ 依次尝试各种格式去解析
    parsed = False
    for fmt in [
        '%Y-%m-%d %H:%M:%S',
        '%Y/%m/%d %H:%M:%S',
        '%Y-%m-%d %H:%M',
        '%Y/%m/%d %H:%M',
    ]:
        try:
            df[date_col] = pd.to_datetime(
                df[date_col],
                format=fmt,     # 严格匹配
                errors='raise'  # 抛错就切换下一个 fmt
            )
            parsed = True
            break
        except Exception:
            continue

    # ④ 如果上面都没能解析，再宽松一把
    if not parsed:
        df[date_col] = pd.to_datetime(df[date_col], errors='coerce')

    # ⑤ 把分钟/秒都砍掉，保留到「小时」粒度
    df[date_col] = df[date_col].dt.floor('h').dt.tz_localize(None)

    # ⑥ 把这一列重命名为 DATE
    df = df.rename(columns={date_col: 'DATE'})

    # （可选）如果你想让 DATE 作为索引：
    # df = df.set_index('DATE')

    return df



raw_dfs = {}
for name, path in file_paths.items():
    raw_dfs[name] = read_and_clean(path)
    # print(raw_dfs[name].columns)

# ===== 資料結構：取代 all_data =====
class AssetGroupLite:
    def __init__(self, name, df):
        self.name = name
        df = df.copy()
        df['DATE'] = pd.to_datetime(df['DATE'])
        self.raw = df.set_index('DATE').sort_index()

    @cached_property
    def _close_cols(self):
        return [c for c in self.raw.columns if c.endswith('_CLOSE')]

    @cached_property
    def _vol_cols(self):
        return [c for c in self.raw.columns if c.endswith('_VOLUME')]

    @cached_property
    def close_ln(self):
        if not self._close_cols:
            return self.raw.iloc[[]]
        return np.log(self.raw[self._close_cols]).rename(
            columns=lambda x: x.replace('_CLOSE', '_CLOSE_ln')
        )

    @cached_property
    def close_ln_ret(self):
        if not self._close_cols:
            return self.raw.iloc[[]]
        logp = np.log(self.raw[self._close_cols])
        return (
            logp.diff()
               .rename(columns=lambda x: x.replace('_CLOSE', '_CLOSE_ln_ret'))
               .dropna(how='all')
        )

    @cached_property
    def close_arith_ret(self):
        if not self._close_cols:
            return self.raw.iloc[[]]
        return (
            self.raw[self._close_cols].pct_change()
                .rename(columns=lambda x: x.replace('_CLOSE', '_CLOSE_arith_ret'))
                .dropna(how='all')
        )

class DataRepository:
    REQUIRED_INDEX = "DATE"

    def __init__(self, raw_dfs: dict, check_schema: bool = True):
        self.groups = {}
        for name, df in raw_dfs.items():
            if check_schema:
                assert 'DATE' in df.columns, f"{name}: 缺少 DATE 欄"
            self.groups[name] = AssetGroupLite(name, df)

    # 補上 group()，方便外部與內部呼叫
    def group(self, name: str) -> AssetGroupLite:
        if name not in self.groups:
            raise KeyError(f"Group '{name}' 不存在。可用群組：{list(self.groups.keys())}")
        return self.groups[name]

    def series(self, group: str, series_name: str) -> pd.Series:
        g = self.group(group)
        # 加上 raw → 能抓 OHLCV、IS_TRADING 等原始欄
        search_order = ['close_ln_ret', 'close_arith_ret', 'close_ln', 'raw']

        # 1) 直接命中
        for key in search_order:
            tbl = getattr(g, key)
            if series_name in tbl.columns:
                return tbl[series_name]

        # 2) 容錯：只給 base symbol，自動補候選
        base = (series_name
                .replace('_CLOSE', '')
                .replace('_OPEN', '')
                .replace('_HIGH', '')
                .replace('_LOW', '')
                .replace('_VOLUME', '')
                .replace('_IS_TRADING', '')
                .replace('_CLOSE_ln_ret', '')
                .replace('_CLOSE_arith_ret', '')
                .replace('_CLOSE_ln', ''))
        candidates = [
            f'{base}_CLOSE_ln_ret',
            f'{base}_CLOSE_arith_ret',
            f'{base}_CLOSE_ln',
            f'{base}_OPEN',
            f'{base}_HIGH',
            f'{base}_LOW',
            f'{base}_CLOSE',
            f'{base}_VOLUME',
            f'{base}_IS_TRADING'
        ]
        for cand in candidates:
            for key in search_order:
                tbl = getattr(g, key)
                if cand in tbl.columns:
                    return tbl[cand]

        raise KeyError(f"{group}: 找不到 {series_name} 或候選 {candidates}")

    # 取整張表
    def table(self, group: str, table_name: str) -> pd.DataFrame:
        g = self.group(group)
        if not hasattr(g, table_name):
            raise KeyError(f"{group}: 無表 '{table_name}'。可用表：['close_ln_ret','close_arith_ret','close_ln']")
        return getattr(g, table_name)

    # 若要直接拿 raw 的原始價/量欄位（例如 *_CLOSE 或 *_VOLUME）
    def raw_series(self, group: str, raw_col: str) -> pd.Series:
        g = self.group(group)
        if raw_col not in g.raw.columns:
            raise KeyError(f"{group}: raw 中沒有欄位 {raw_col}")
        return g.raw[raw_col]

repo = DataRepository(raw_dfs)
print("repo success")

Num GPUs: 0 []
是否可用GPU: False
使用中的裝置: []
✓ GPU 初始化流程完成
repo success


In [19]:
# 1. 定義檔案路徑
garch_file_paths = {
    "garch_params_1": "./garch_data/garch_params_by_day_1.csv"
}

def garch_find_date_col(df):
    for col in df.columns:
        if 'date' in col.lower():
            return col
    return df.columns[0]

def garch_read_and_clean(file):
    # ① 正确地读 CSV，不要写 (index=True)
    df = pd.read_csv(file)

    # ② 找到原始时间列名
    date_col = garch_find_date_col(df)

    # ③ 依次尝试各种格式去解析
    parsed = False
    for fmt in [
        '%Y-%m-%d %H:%M:%S',
        '%Y/%m/%d %H:%M:%S',
        '%Y-%m-%d %H:%M',
        '%Y/%m/%d %H:%M',
    ]:
        try:
            df[date_col] = pd.to_datetime(
                df[date_col],
                format=fmt,     # 严格匹配
                errors='raise'  # 抛错就切换下一个 fmt
            )
            parsed = True
            break
        except Exception:
            continue

    # ④ 如果上面都没能解析，再宽松一把
    if not parsed:
        df[date_col] = pd.to_datetime(df[date_col], errors='coerce')

    # ⑤ 把分钟/秒都砍掉，保留到「小时」粒度
    df[date_col] = df[date_col].dt.floor('h').dt.tz_localize(None)

    # ⑥ 把这一列重命名为 DATE
    df = df.rename(columns={date_col: 'DATE'})

    # （可选）如果你想让 DATE 作为索引：
    # df = df.set_index('DATE')

    return df

garch_dfs = {}
for name, path in garch_file_paths.items():
    garch_dfs[name] = garch_read_and_clean(path)
    print(garch_dfs[name].columns)

Index(['DATE', 'sigma_next', 'omega', 'alpha', 'beta', 'shape', '_fallback',
       'vxreg1_p', 'vxreg2_p', '_conv', '_llh', 'ab'],
      dtype='object')


In [23]:
# ------------------ Features ------------------
def build_feature_df(repo: DataRepository, group: str, symbol: str) -> pd.DataFrame:
    s_open  = repo.raw_series(group, f'{symbol}_OPEN').asfreq('D')
    s_high  = repo.raw_series(group, f'{symbol}_HIGH').asfreq('D')
    s_low   = repo.raw_series(group, f'{symbol}_LOW').asfreq('D')
    s_close = repo.raw_series(group, f'{symbol}_CLOSE').asfreq('D')
    s_vol   = repo.raw_series(group, f'{symbol}_VOLUME').asfreq('D')
    s_flag = repo.raw_series(group, f'{symbol}_IS_TRADING').asfreq('D')
    s_lnrt  = repo.series(group, f'{symbol}_CLOSE_ln_ret').asfreq('D')
    s_ln  = repo.series(group, f'{symbol}_CLOSE_ln').asfreq('D')
    df = pd.concat([
        s_lnrt.rename(f'{symbol}_LN_RET'),
        s_open.rename(f'{symbol}_OPEN'),
        s_high.rename(f'{symbol}_HIGH'),
        s_low.rename(f'{symbol}_LOW'),
        s_close.rename(f'{symbol}_CLOSE'),
        s_vol.rename(f'{symbol}_VOLUME'),
        s_flag.rename(f'{symbol}_IS_TRADING'),
        s_ln.rename(f'{symbol}_CLOSE_LN')
        ], axis=1)
    df = df.apply(pd.to_numeric, errors='coerce')
    return df.dropna(how='any')



# ------------------ Config ------------------
ASSET_SYMBOL_ES1 = 'ES1'
ASSET_SYMBOL_VIX = 'VIX'
GROUP_DAY   = 'stock_day'
TARGET_START_STR = '2005-06-01'
TARGET_END_STR   = '2025-06-30'

# # 將輸出在特定的區間
# EXPORT_START = pd.to_datetime('2005-06-01')
# EXPORT_END   = pd.to_datetime('2025-06-30')

# GARCH_WINDOW_DAY = 252
VOL_WINDOW_DAY = 20


feat_ES1 = build_feature_df(repo, GROUP_DAY, ASSET_SYMBOL_ES1)
feat_VIX = build_feature_df(repo, GROUP_DAY, ASSET_SYMBOL_VIX)

df_day = feat_ES1.join(feat_VIX, how='left')

# --- 讓索引成為 datetime（很重要） ---
df_day.index = pd.to_datetime(df_day.index, errors='coerce')
df_day = df_day.sort_index()
df_day = df_day.loc[df_day['ES1_IS_TRADING'] == 1]

# print(df_day.head())
# print(df_day.columns)

df_day.to_csv("./filtered_output/df_day.csv", index=True)


PRED_START = pd.to_datetime(TARGET_START_STR)
PRED_END   = pd.to_datetime(TARGET_END_STR)

# 把回測期限制在資料範圍內
data_start = df_day.index.min()
data_end   = df_day.index.max()
if PRED_START < data_start: PRED_START = data_start
if PRED_END   > data_end:   PRED_END   = data_end

# 若 PRED_START 不是可用交易日，推到 >= PRED_START 的第一個交易日
try:
    PRED_START = df_day.index[df_day.index.searchsorted(PRED_START)]
except Exception:
    # 若整段都沒資料，直接報錯
    raise RuntimeError("資料期間與回測期間沒有交集，請調整 TARGET_START/END。")


first_needed = PRED_START - pd.Timedelta(days= VOL_WINDOW_DAY)
es1_min = df_day.index.min()
ES1_close_series = df_day['ES1_CLOSE']
if es1_min > first_needed:
    raise RuntimeError(f"Insufficient history: need <= {first_needed}, have from {ES1_close_series.index.min()}")



# ------------------ Model ------------------


# 1) 準備母表（確保排序與期間）
# 全歷史的交易日索引（只留 ES1 開市）
dates_all = df_day.index
# print(dates_all)

# 回測區間（仍然要取，決定哪些 t 需要預測）
mask_period = (df_day.index >= PRED_START) & (df_day.index <= PRED_END)
dates_period = df_day.index[mask_period]

# 2) 方便取用的短名
feat_df = df_day  # 與舊程式一致
rows = []




# 將 GARCH 數據加進 feat_df
garch_df = next(iter(garch_dfs.values())).copy()

# --- 1) 整理 garch_df：DATE -> index，只留兩欄，並轉成「日」粒度 ---
garch_small = (
    garch_df[["DATE", "sigma_next", "shape"]]
      .assign(DATE=lambda d: pd.to_datetime(d["DATE"], errors="coerce").dt.floor("D"))  # ✅ 改這行
      .dropna(subset=["DATE"])
      .set_index("DATE")
      .sort_index()
)

# 若同一天有重複（有時候資料會），保留最後一筆
garch_small = garch_small[~garch_small.index.duplicated(keep="last")]

# --- 2) feat_df 的 index 也轉成「日」粒度（避免含時分秒對不到）---
feat_df = feat_df.copy()
feat_df.index = pd.to_datetime(feat_df.index, errors="coerce").floor("D")  # ✅ 改這行
feat_df = feat_df.sort_index()

# --- 3) 以 feat_df 為主表做 join（DATE 對齊）---
feat_df = feat_df.join(garch_small, how="left")




import numpy as np
import pandas as pd

# =========================================================
# 參數設定
# =========================================================
VOL_WINDOW_DAY = 20
VOL_WINDOW_DAY_10 = 10
VOL_WINDOW_DAY_5 = 5
EWMA_LAMBDA = 0.94

ret_col   = 'ES1_LN_RET'
open_col  = 'ES1_OPEN'
high_col  = 'ES1_HIGH'
low_col   = 'ES1_LOW'
close_col = 'ES1_CLOSE'

# =========================================================
# 0. 複製資料並排序
# =========================================================
vol_df = feat_df.copy()
vol_df = vol_df.sort_index()

required_cols = [ret_col, open_col, high_col, low_col, close_col]
missing_cols = [c for c in required_cols if c not in vol_df.columns]
if missing_cols:
    raise KeyError(f"缺少必要欄位: {missing_cols}")

for c in required_cols:
    vol_df[c] = pd.to_numeric(vol_df[c], errors='coerce')

# =========================================================
# 1. 基本變數
# =========================================================
# close-to-close log return
vol_df['ret'] = vol_df[ret_col]

# OHLC 對數比率
vol_df['log_oc_prev'] = np.log(vol_df[open_col] / vol_df[close_col].shift(1))  # o_t = ln(O_t / C_{t-1})
vol_df['log_ho']      = np.log(vol_df[high_col] / vol_df[open_col])            # u_t = ln(H_t / O_t)
vol_df['log_lo']      = np.log(vol_df[low_col]  / vol_df[open_col])            # d_t = ln(L_t / O_t)
vol_df['log_co']      = np.log(vol_df[close_col] / vol_df[open_col])           # c_t = ln(C_t / O_t)
vol_df['log_hl']      = np.log(vol_df[high_col] / vol_df[low_col])             # ln(H_t / L_t)

# =========================================================
# 2. SMA volatility
# 目標定義：包含 r_t，因此是第 t 天的 target proxy
# =========================================================
rolling_mean_20 = vol_df['ret'].rolling(
    window=VOL_WINDOW_DAY,
    min_periods=VOL_WINDOW_DAY
).mean()

rolling_mean_sq_20 = (vol_df['ret'] ** 2).rolling(
    window=VOL_WINDOW_DAY,
    min_periods=VOL_WINDOW_DAY
).mean()

vol_df['sma_var_20'] = rolling_mean_sq_20 - (rolling_mean_20 ** 2)
vol_df['sma_var_20'] = vol_df['sma_var_20'].clip(lower=0)
vol_df['sma_vol_20'] = np.sqrt(vol_df['sma_var_20'])

# 10 day
rolling_mean_10 = vol_df['ret'].rolling(
    window=VOL_WINDOW_DAY_10,
    min_periods=VOL_WINDOW_DAY_10
).mean()

rolling_mean_sq_10 = (vol_df['ret'] ** 2).rolling(
    window=VOL_WINDOW_DAY_10,
    min_periods=VOL_WINDOW_DAY_10
).mean()

vol_df['sma_var_10'] = rolling_mean_sq_10 - (rolling_mean_10 ** 2)
vol_df['sma_var_10'] = vol_df['sma_var_10'].clip(lower=0)
vol_df['sma_vol_10'] = np.sqrt(vol_df['sma_var_10'])

# 5 day
rolling_mean_5 = vol_df['ret'].rolling(
    window=VOL_WINDOW_DAY_5,
    min_periods=VOL_WINDOW_DAY_5
).mean()

rolling_mean_sq_5 = (vol_df['ret'] ** 2).rolling(
    window=VOL_WINDOW_DAY_5,
    min_periods=VOL_WINDOW_DAY_5
).mean()

vol_df['sma_var_5'] = rolling_mean_sq_5 - (rolling_mean_5 ** 2)
vol_df['sma_var_5'] = vol_df['sma_var_5'].clip(lower=0)
vol_df['sma_vol_5'] = np.sqrt(vol_df['sma_var_5'])

# ----------------------------
#  SMA predict (傳統版：同一窗口平均數)
# ----------------------------

ret_lag1 = vol_df['ret'].shift(1)

vol_df['sma_var_20_predict'] = ret_lag1.rolling(
    window=VOL_WINDOW_DAY,
    min_periods=VOL_WINDOW_DAY
).var(ddof=0)

vol_df['sma_vol_20_predict'] = np.sqrt(vol_df['sma_var_20_predict'])

vol_df['sma_var_10_predict'] = ret_lag1.rolling(
    window=VOL_WINDOW_DAY_10,
    min_periods=VOL_WINDOW_DAY_10
).var(ddof=0)

vol_df['sma_vol_10_predict'] = np.sqrt(vol_df['sma_var_10_predict'])

vol_df['sma_var_5_predict'] = ret_lag1.rolling(
    window=VOL_WINDOW_DAY_5,
    min_periods=VOL_WINDOW_DAY_5
).var(ddof=0)

vol_df['sma_vol_5_predict'] = np.sqrt(vol_df['sma_var_5_predict'])

# 如果你要改成 sample variance 版本，可改用這段：
# vol_df['sma_var_20'] = vol_df['ret'].rolling(
#     window=VOL_WINDOW_DAY,
#     min_periods=VOL_WINDOW_DAY
# ).var(ddof=1)
# vol_df['sma_var_20'] = vol_df['sma_var_20'].clip(lower=0)
# vol_df['sma_vol_20'] = np.sqrt(vol_df['sma_var_20'])

# =========================================================
# 3. EWMA volatility
# 目標定義：第 t 天用 r_t^2 更新
# =========================================================
vol_df['ewma_var'] = np.nan

valid_idx = vol_df.index[vol_df['ret'].notna()]

if len(valid_idx) > 0:
    first_idx = valid_idx[0]
    vol_df.loc[first_idx, 'ewma_var'] = vol_df.loc[first_idx, 'ret'] ** 2

    for i in range(1, len(valid_idx)):
        idx = valid_idx[i]
        prev_idx = valid_idx[i - 1]

        prev_var = vol_df.loc[prev_idx, 'ewma_var']
        curr_r2 = vol_df.loc[idx, 'ret'] ** 2

        vol_df.loc[idx, 'ewma_var'] = (
            EWMA_LAMBDA * prev_var + (1 - EWMA_LAMBDA) * curr_r2
        )

vol_df['ewma_var'] = vol_df['ewma_var'].clip(lower=0)
vol_df['ewma_vol'] = np.sqrt(vol_df['ewma_var'])

# =========================================================
# 4. GK volatility
# 單日 GK + 20日 rolling 平均
# =========================================================
vol_df['gk_var_daily'] = (
    0.5 * (vol_df['log_hl'] ** 2)
    - (2 * np.log(2) - 1) * (vol_df['log_co'] ** 2)
)
vol_df['gk_var_daily'] = vol_df['gk_var_daily'].clip(lower=0)
vol_df['gk_vol_daily'] = np.sqrt(vol_df['gk_var_daily'])

# 20 day
vol_df['gk_var_20'] = vol_df['gk_var_daily'].rolling(
    window=VOL_WINDOW_DAY,
    min_periods=VOL_WINDOW_DAY
).mean()
vol_df['gk_var_20'] = vol_df['gk_var_20'].clip(lower=0)
vol_df['gk_vol_20'] = np.sqrt(vol_df['gk_var_20'])

# 10 day
vol_df['gk_var_10'] = vol_df['gk_var_daily'].rolling(
    window=VOL_WINDOW_DAY_10,
    min_periods=VOL_WINDOW_DAY_10
).mean()
vol_df['gk_var_10'] = vol_df['gk_var_10'].clip(lower=0)
vol_df['gk_vol_10'] = np.sqrt(vol_df['gk_var_10'])

# 5 day
vol_df['gk_var_5'] = vol_df['gk_var_daily'].rolling(
    window=VOL_WINDOW_DAY_5,
    min_periods=VOL_WINDOW_DAY_5
).mean()
vol_df['gk_var_5'] = vol_df['gk_var_5'].clip(lower=0)
vol_df['gk_vol_5'] = np.sqrt(vol_df['gk_var_5'])


# =========================================================
# 5. RS volatility
# 單日 RS + 20日 rolling 平均
# =========================================================
vol_df['rs_var_daily'] = (
    vol_df['log_ho'] * (vol_df['log_ho'] - vol_df['log_co'])
    + vol_df['log_lo'] * (vol_df['log_lo'] - vol_df['log_co'])
)
vol_df['rs_var_daily'] = vol_df['rs_var_daily'].clip(lower=0)
vol_df['rs_vol_daily'] = np.sqrt(vol_df['rs_var_daily'])

# 20 day
vol_df['rs_var_20'] = vol_df['rs_var_daily'].rolling(
    window=VOL_WINDOW_DAY,
    min_periods=VOL_WINDOW_DAY
).mean()
vol_df['rs_var_20'] = vol_df['rs_var_20'].clip(lower=0)
vol_df['rs_vol_20'] = np.sqrt(vol_df['rs_var_20'])

# 10 day
vol_df['rs_var_10'] = vol_df['rs_var_daily'].rolling(
    window=VOL_WINDOW_DAY_10,
    min_periods=VOL_WINDOW_DAY_10
).mean()
vol_df['rs_var_10'] = vol_df['rs_var_10'].clip(lower=0)
vol_df['rs_vol_10'] = np.sqrt(vol_df['rs_var_10'])

# 5 day
vol_df['rs_var_5'] = vol_df['rs_var_daily'].rolling(
    window=VOL_WINDOW_DAY_5,
    min_periods=VOL_WINDOW_DAY_5
).mean()
vol_df['rs_var_5'] = vol_df['rs_var_5'].clip(lower=0)
vol_df['rs_vol_5'] = np.sqrt(vol_df['rs_var_5'])
# =========================================================
# 6. YZ volatility
# 截至第 t 天的 window-based YZ
# =========================================================

yz_windows = [VOL_WINDOW_DAY, VOL_WINDOW_DAY_10, VOL_WINDOW_DAY_5]

for w in yz_windows:
    # 6.1 開盤跳空變異數 V_O
    yz_open_mean = vol_df['log_oc_prev'].rolling(
        window=w,
        min_periods=w
    ).mean()

    yz_open_sq_mean = (vol_df['log_oc_prev'] ** 2).rolling(
        window=w,
        min_periods=w
    ).mean()

    vol_df[f'yz_vo_{w}'] = yz_open_sq_mean - (yz_open_mean ** 2)
    vol_df[f'yz_vo_{w}'] = vol_df[f'yz_vo_{w}'].clip(lower=0)

    # 6.2 收盤內部變異數 V_C
    yz_close_mean = vol_df['log_co'].rolling(
        window=w,
        min_periods=w
    ).mean()

    yz_close_sq_mean = (vol_df['log_co'] ** 2).rolling(
        window=w,
        min_periods=w
    ).mean()

    vol_df[f'yz_vc_{w}'] = yz_close_sq_mean - (yz_close_mean ** 2)
    vol_df[f'yz_vc_{w}'] = vol_df[f'yz_vc_{w}'].clip(lower=0)

    # 6.3 RS 成分平均 V_RS
    vol_df[f'yz_vrs_{w}'] = vol_df['rs_var_daily'].rolling(
        window=w,
        min_periods=w
    ).mean()
    vol_df[f'yz_vrs_{w}'] = vol_df[f'yz_vrs_{w}'].clip(lower=0)

    # 6.4 YZ 權重 k
    yz_k = 0.34 / (1.34 + (w + 1) / (w - 1))

    # 6.5 YZ 總變異數與波動率
    vol_df[f'yz_var_{w}'] = (
        vol_df[f'yz_vo_{w}']
        + yz_k * vol_df[f'yz_vc_{w}']
        + (1 - yz_k) * vol_df[f'yz_vrs_{w}']
    )
    vol_df[f'yz_var_{w}'] = vol_df[f'yz_var_{w}'].clip(lower=0)
    vol_df[f'yz_vol_{w}'] = np.sqrt(vol_df[f'yz_var_{w}'])

# =========================================================
# 7. GARCH 欄位整理
# =========================================================
if 'sigma_next' in vol_df.columns:
    vol_df['garch_vol'] = pd.to_numeric(vol_df['sigma_next'], errors='coerce')
    vol_df['garch_var'] = vol_df['garch_vol'] ** 2
else:
    vol_df['garch_vol'] = np.nan
    vol_df['garch_var'] = np.nan

# =========================================================
# 8. 篩選目標區間
# =========================================================
target_start = pd.to_datetime(TARGET_START_STR)
target_end = pd.to_datetime(TARGET_END_STR)

vol_result_df = vol_df.loc[
    (vol_df.index >= target_start) & (vol_df.index <= target_end)
].copy()

vol_result_df = vol_result_df[
    ~vol_result_df.index.duplicated(keep='last')
]

# =========================================================
# 9. 輸出欄位整理
# =========================================================
final_cols = [
    ret_col,close_col,

    # SMA
    'sma_var_20', 'sma_vol_20','sma_vol_20_predict',
    'sma_var_10', 'sma_vol_10','sma_vol_10_predict',
    'sma_var_5', 'sma_vol_5','sma_vol_5_predict',

    # EWMA
    'ewma_var', 'ewma_vol',

    # GK
    'gk_var_daily', 'gk_vol_daily','gk_var_20', 'gk_vol_20',
    'gk_var_10', 'gk_vol_10',
    'gk_var_5', 'gk_vol_5',

    # RS
    'rs_var_daily', 'rs_vol_daily','rs_var_20', 'rs_vol_20',
    'rs_var_10', 'rs_vol_10',
    'rs_var_5', 'rs_vol_5',

    # YZ
    'yz_vo_20', 'yz_vc_20', 'yz_vrs_20',
    'yz_var_20', 'yz_vol_20','yz_vol_10','yz_vol_5',


    # GARCH
    'garch_var', 'garch_vol'
]

if 'shape' in vol_result_df.columns:
    final_cols.append('shape')

vol_result_df = vol_result_df[final_cols].copy()
vol_result_df = vol_result_df.loc[
    :, ~vol_result_df.columns.duplicated()
].copy()

vol_result_df = vol_result_df.reset_index().rename(
    columns={'index': 'DATE'}
)

# =========================================================
# 10. 檢查輸出
# =========================================================
print(vol_result_df.head())
print(vol_result_df.tail())
print(vol_result_df.columns.tolist())
print(vol_result_df.shape)

vol_result_df.to_csv(
    "./garch_data/es1_volatility_all_methods.csv",
    index=False
)
vol_result_df.to_csv(
    "./real_volatility_multi_var/es1_volatility_all_methods.csv",
    index=False
)

        DATE  ES1_LN_RET  ES1_CLOSE  sma_var_20  sma_vol_20  \
0 2005-06-01    0.007520    1201.25    0.000036    0.006021   
1 2005-06-02    0.003324    1205.25    0.000034    0.005858   
2 2005-06-03   -0.005616    1198.50    0.000036    0.006041   
3 2005-06-06   -0.000417    1198.00    0.000036    0.005975   
4 2005-06-07    0.000834    1199.00    0.000034    0.005871   

   sma_vol_20_predict  sma_var_10  sma_vol_10  sma_vol_10_predict  sma_var_5  \
0            0.005864    0.000022    0.004677            0.004624   0.000024   
1            0.006021    0.000016    0.003986            0.004677   0.000023   
2            0.005858    0.000020    0.004431            0.003986   0.000029   
3            0.006041    0.000020    0.004419            0.004431   0.000029   
4            0.005975    0.000017    0.004176            0.004419   0.000019   

   ...      yz_vo_20  yz_vc_20  yz_vrs_20  yz_var_20  yz_vol_20  yz_vol_10  \
0  ...  2.178825e-07  0.000035   0.000043   0.000042   0.00649

In [21]:
import os
import numpy as np
import pandas as pd

from scipy.stats import jarque_bera
from statsmodels.stats.diagnostic import acorr_ljungbox
import matplotlib.pyplot as plt

# =========================================================
# 0. Setup
# =========================================================
out_dir = './garch_data'
os.makedirs(out_dir, exist_ok=True)

# 本次使用的 variance 欄位（符合當期 target 定義）
var_cols = [
    'sma_vol_20',
    'sma_vol_10',
    'sma_vol_5',

    'ewma_vol',

    'gk_vol_daily',
    'gk_vol_20',
    'gk_vol_10',
    'gk_vol_5',

    'rs_vol_daily',
    'rs_vol_20',
    'rs_vol_10',
    'rs_vol_5',

    'yz_vol_20','yz_vol_10','yz_vol_5',
    'garch_vol',

    'sma_vol_20_predict',
    'sma_vol_10_predict',
    'sma_vol_5_predict'
]

# 只保留實際存在的欄位
var_cols = [c for c in var_cols if c in vol_result_df.columns]

if 'DATE' not in vol_result_df.columns:
    raise KeyError("vol_result_df 缺少 DATE 欄位")

if len(var_cols) == 0:
    raise ValueError("vol_result_df 中找不到任何指定的 variance 欄位")

# 建立分析資料
var_df = vol_result_df[['DATE'] + var_cols].copy()
var_df['DATE'] = pd.to_datetime(var_df['DATE'], errors='coerce')
var_df = var_df.sort_values('DATE').dropna(subset=['DATE']).copy()

# =========================================================
# 1. Descriptive statistics
# =========================================================
desc_stats = var_df[var_cols].agg(['count', 'mean', 'std', 'min', 'max']).T

quantiles = var_df[var_cols].quantile(
    [0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99]
).T
quantiles.columns = ['q01', 'q05', 'q25', 'q50', 'q75', 'q95', 'q99']

desc_stats['skew'] = var_df[var_cols].skew()
desc_stats['kurt'] = var_df[var_cols].kurt()

summary_stats_df = pd.concat([desc_stats, quantiles], axis=1)
summary_stats_df = summary_stats_df[
    [
        'count', 'mean', 'std', 'min',
        'q01', 'q05', 'q25', 'q50', 'q75', 'q95', 'q99',
        'max', 'skew', 'kurt'
    ]
]

# =========================================================
# 2. Jarque-Bera
# =========================================================
jb_rows = []

for col in var_cols:
    s = var_df[col].dropna()

    if len(s) > 0:
        jb_stat, jb_p = jarque_bera(s)

        jb_rows.append({
            'variable': col,
            'n': len(s),
            'jb_stat': jb_stat,
            'jb_pvalue': jb_p,
            'normal_at_5pct': 'Yes' if jb_p >= 0.05 else 'No'
        })

jb_df = pd.DataFrame(jb_rows)

# =========================================================
# 3. Correlations
# =========================================================
pearson_corr_df = var_df[var_cols].corr(method='pearson')
spearman_corr_df = var_df[var_cols].corr(method='spearman')

# =========================================================
# 4. Ljung-Box
# =========================================================
lb_rows = []

for col in var_cols:
    s = var_df[col].dropna()

    if len(s) > 30:
        lb_5 = acorr_ljungbox(s, lags=[5], return_df=True)
        lb_10 = acorr_ljungbox(s, lags=[10], return_df=True)
        lb_20 = acorr_ljungbox(s, lags=[20], return_df=True)

        lb_rows.append({
            'variable': col,
            'acf_lag1': s.autocorr(lag=1),
            'acf_lag5': s.autocorr(lag=5),
            'acf_lag10': s.autocorr(lag=10),
            'lb_stat_5': lb_5['lb_stat'].iloc[0],
            'lb_pvalue_5': lb_5['lb_pvalue'].iloc[0],
            'lb_stat_10': lb_10['lb_stat'].iloc[0],
            'lb_pvalue_10': lb_10['lb_pvalue'].iloc[0],
            'lb_stat_20': lb_20['lb_stat'].iloc[0],
            'lb_pvalue_20': lb_20['lb_pvalue'].iloc[0]
        })

lb_df = pd.DataFrame(lb_rows)

# =========================================================
# 5. Pairwise difference summary
# =========================================================
pair_rows = []

for i in range(len(var_cols)):
    for j in range(i + 1, len(var_cols)):
        c1 = var_cols[i]
        c2 = var_cols[j]

        tmp = var_df[[c1, c2]].dropna().copy()
        if len(tmp) == 0:
            continue

        diff = tmp[c1] - tmp[c2]

        pair_rows.append({
            'var1': c1,
            'var2': c2,
            'n': len(diff),
            'mean_diff': diff.mean(),
            'std_diff': diff.std(),
            'mae': diff.abs().mean(),
            'rmse': np.sqrt(np.mean(diff ** 2)),
            'corr_pearson': tmp[c1].corr(tmp[c2], method='pearson'),
            'corr_spearman': tmp[c1].corr(tmp[c2], method='spearman')
        })

pairwise_diff_df = pd.DataFrame(pair_rows)

# =========================================================
# 6. Console output
# =========================================================
print("\n[Summary stats]")
print(summary_stats_df.round(8))

print("\n[Jarque-Bera]")
print(jb_df.round(6))

print("\n[Pearson correlation]")
print(pearson_corr_df.round(4))

print("\n[Spearman correlation]")
print(spearman_corr_df.round(4))

print("\n[Ljung-Box]")
print(lb_df.round(6))

print("\n[Pairwise difference summary]")
print(pairwise_diff_df.round(8))

# =========================================================
# 7. Save tables
# =========================================================
summary_stats_df.to_csv(
    f'{out_dir}/summary_stats_var.csv',
    encoding='utf-8-sig'
)
jb_df.to_csv(
    f'{out_dir}/jarque_bera_var.csv',
    index=False,
    encoding='utf-8-sig'
)
pearson_corr_df.to_csv(
    f'{out_dir}/pearson_corr_var.csv',
    encoding='utf-8-sig'
)
spearman_corr_df.to_csv(
    f'{out_dir}/spearman_corr_var.csv',
    encoding='utf-8-sig'
)
lb_df.to_csv(
    f'{out_dir}/ljung_box_var.csv',
    index=False,
    encoding='utf-8-sig'
)
pairwise_diff_df.to_csv(
    f'{out_dir}/pairwise_diff_var.csv',
    index=False,
    encoding='utf-8-sig'
)

print("\nAll variance statistics tables saved.")

# =========================================================
# 8. Plot: individual figures
# =========================================================
for col in var_cols:
    plot_df = var_df[['DATE', col]].dropna().copy()

    plt.figure(figsize=(12, 5))
    plt.plot(plot_df['DATE'], plot_df[col], linewidth=1)
    plt.title(col)
    plt.xlabel('Date')
    plt.ylabel(col)
    plt.tight_layout()
    plt.savefig(f'{out_dir}/{col}.png', dpi=200, bbox_inches='tight')
    plt.close()

# =========================================================
# 9. Plot: grouped comparison figures
# =========================================================

# 9.1 單日 realized 型
group_daily = [c for c in ['gk_var_daily', 'rs_var_daily'] if c in var_df.columns]

if len(group_daily) > 0:
    plt.figure(figsize=(12, 5))
    for col in group_daily:
        plot_df = var_df[['DATE', col]].dropna().copy()
        plt.plot(plot_df['DATE'], plot_df[col], linewidth=1, label=col)

    plt.title('Daily realized-type variance comparison')
    plt.xlabel('Date')
    plt.ylabel('Variance')
    plt.legend()
    plt.tight_layout()
    plt.savefig(f'{out_dir}/daily_realized_variance_comparison.png', dpi=200, bbox_inches='tight')
    plt.close()

# 9.2 Rolling / smoothed / model-based 型
group_smooth = [c for c in ['sma_var_20', 'ewma_var', 'yz_var_20', 'garch_var'] if c in var_df.columns]

if len(group_smooth) > 0:
    plt.figure(figsize=(12, 5))
    for col in group_smooth:
        plot_df = var_df[['DATE', col]].dropna().copy()
        plt.plot(plot_df['DATE'], plot_df[col], linewidth=1, label=col)

    plt.title('Smoothed / model-based variance comparison')
    plt.xlabel('Date')
    plt.ylabel('Variance')
    plt.legend()
    plt.tight_layout()
    plt.savefig(f'{out_dir}/smoothed_model_variance_comparison.png', dpi=200, bbox_inches='tight')
    plt.close()

# 9.3 全部一起比較
plt.figure(figsize=(14, 6))
for col in var_cols:
    plot_df = var_df[['DATE', col]].dropna().copy()
    plt.plot(plot_df['DATE'], plot_df[col], linewidth=1, label=col)

plt.title('All variance measures comparison')
plt.xlabel('Date')
plt.ylabel('Variance')
plt.legend()
plt.tight_layout()
plt.savefig(f'{out_dir}/all_variance_measures_comparison.png', dpi=200, bbox_inches='tight')
plt.close()

print(f'\n圖已存到: {out_dir}')


[Summary stats]
                     count      mean       std       min       q01       q05  \
sma_vol_20          5072.0  0.009781  0.006919  0.001821  0.002917  0.003792   
sma_vol_10          5072.0  0.009307  0.007148  0.001021  0.002160  0.003155   
ewma_vol            5072.0  0.010253  0.006628  0.002665  0.003568  0.004357   
gk_vol_daily        5072.0  0.009229  0.007146  0.001204  0.002251  0.003086   
gk_vol_20           5072.0  0.009933  0.006128  0.002671  0.003885  0.004661   
gk_vol_10           5072.0  0.009778  0.006374  0.002202  0.003552  0.004412   
gk_vol_5            5072.0  0.009639  0.006583  0.001706  0.003077  0.003983   
rs_vol_daily        5072.0  0.009114  0.007221  0.000979  0.002057  0.002977   
rs_vol_20           5072.0  0.009923  0.006058  0.002710  0.003953  0.004712   
rs_vol_10           5072.0  0.009757  0.006324  0.002183  0.003586  0.004423   
rs_vol_5            5072.0  0.009598  0.006563  0.001726  0.003128  0.003986   
yz_vol_20           507

In [25]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import norm
from scipy.stats import t as tdist
from scipy.stats import chi2


# =========================================================
# 0. 函數：自動計算 VaR / ES
# =========================================================
def compute_var_es_auto(
    mu,
    sigma,
    alpha,
    nu=None,
    price_base=None,
    standardized_t=True,
    nu_clip=None
):
    if not (0.0 < alpha < 0.5):
        raise ValueError("alpha 應位於 (0, 0.5)。")

    mu_s = pd.Series(mu).astype(float)
    sig_s = pd.Series(sigma).astype(float).clip(lower=0.0)

    idx = mu_s.index.union(sig_s.index)
    mu_s = mu_s.reindex(idx)
    sig_s = sig_s.reindex(idx)

    nu_s = None
    if nu is not None:
        nu_s = pd.Series(nu).reindex(idx).astype(float)

    price_s = pd.Series(price_base).reindex(idx).astype(float) if price_base is not None else None

    if nu_s is None:
        is_t = pd.Series(False, index=idx)
    else:
        is_t = nu_s.notna()

    is_norm = ~is_t

    VaR_ret = pd.Series(np.nan, index=idx, dtype=float)
    ES_ret = pd.Series(np.nan, index=idx, dtype=float)

    # ===== Normal =====
    if is_norm.any():
        z_alpha = norm.ppf(alpha)
        phi = norm.pdf(z_alpha)
        ES_std = -phi / alpha

        VaR_ret.loc[is_norm] = (mu_s + sig_s * z_alpha).loc[is_norm]
        ES_ret.loc[is_norm] = (mu_s + sig_s * ES_std).loc[is_norm]

    # ===== Student-t =====
    if is_t.any():
        nu_use = nu_s.copy()

        invalid = is_t & (nu_use <= 2)
        if invalid.any():
            bad_idx = list(nu_use[invalid].index[:5])
            raise ValueError(f"t 分布計算 ES 需要 nu > 2；前幾個不合法 index：{bad_idx}")

        if nu_clip is not None:
            lo, hi = float(nu_clip[0]), float(nu_clip[1])
            nu_use = nu_use.clip(lower=lo, upper=hi)

        t_alpha = pd.Series(tdist.ppf(alpha, df=nu_use.values), index=idx)
        f_t = pd.Series(tdist.pdf(t_alpha.values, df=nu_use.values), index=idx)

        ES_std = -((nu_use + t_alpha**2) / ((nu_use - 1.0) * alpha)) * f_t

        if standardized_t:
            s = np.sqrt((nu_use - 2.0) / nu_use)
        else:
            s = 1.0

        VaR_std = s * t_alpha
        ES_std = s * ES_std

        VaR_ret.loc[is_t] = (mu_s + sig_s * VaR_std).loc[is_t]
        ES_ret.loc[is_t] = (mu_s + sig_s * ES_std).loc[is_t]

    out = pd.DataFrame({
        'VaR_ret': VaR_ret,
        'ES_ret': ES_ret
    }, index=idx)

    if price_s is not None:
        out['VaR_price'] = price_s * np.exp(out['VaR_ret'])
        out['ES_price'] = price_s * np.exp(out['ES_ret'])

    return out.reindex(mu_s.index)


# =========================================================
# 1. 回測檢定函數
# =========================================================
def kupiec_uc_test(hit_series, p=0.05):
    hit = np.asarray(hit_series, dtype=int)
    n = len(hit)
    x = hit.sum()

    eps = 1e-12
    phat = np.clip(x / n, eps, 1 - eps)

    lr_uc = -2 * (
        ((n - x) * np.log(1 - p) + x * np.log(p))
        - ((n - x) * np.log(1 - phat) + x * np.log(phat))
    )
    p_value = 1 - chi2.cdf(lr_uc, df=1)

    return lr_uc, p_value, x / n


def christoffersen_cc_test(hit_series):
    hit = np.asarray(hit_series, dtype=int)
    n = len(hit)

    n00 = n01 = n10 = n11 = 0
    for i in range(1, n):
        prev_hit = hit[i - 1]
        curr_hit = hit[i]

        if prev_hit == 0 and curr_hit == 0:
            n00 += 1
        elif prev_hit == 0 and curr_hit == 1:
            n01 += 1
        elif prev_hit == 1 and curr_hit == 0:
            n10 += 1
        else:
            n11 += 1

    eps = 1e-12

    pi0 = n01 / (n00 + n01) if (n00 + n01) > 0 else np.nan
    pi1 = n11 / (n10 + n11) if (n10 + n11) > 0 else np.nan
    pi = (n01 + n11) / (n00 + n01 + n10 + n11)

    pi0 = np.clip(pi0, eps, 1 - eps)
    pi1 = np.clip(pi1, eps, 1 - eps)
    pi = np.clip(pi, eps, 1 - eps)

    lr_ind = -2 * (
        ((n00 + n10) * np.log(1 - pi) + (n01 + n11) * np.log(pi))
        - (
            n00 * np.log(1 - pi0)
            + n01 * np.log(pi0)
            + n10 * np.log(1 - pi1)
            + n11 * np.log(pi1)
        )
    )
    p_ind = 1 - chi2.cdf(lr_ind, df=1)

    return lr_ind, p_ind, n00, n01, n10, n11


# =========================================================
# 2. Setup
# =========================================================
EXPORT_START = pd.to_datetime('2005-08-01')
EXPORT_END   = pd.to_datetime('2025-06-30')

alpha = 0.05
STD_T = True
NU_CLIP = (6, 10)

vol_cols = [
    'sma_vol_20',
    'sma_vol_10',
    'sma_vol_5',

    'ewma_vol',

    'gk_vol_daily',
    'gk_vol_20',
    'gk_vol_10',
    'gk_vol_5',

    'rs_vol_daily',
    'rs_vol_20',
    'rs_vol_10',
    'rs_vol_5',

    'yz_vol_20','yz_vol_10','yz_vol_5',
    'garch_vol',

    'sma_vol_20_predict',
    'sma_vol_10_predict',
    'sma_vol_5_predict'
]

output_dir_real = "./real_volatility_multi_var"
os.makedirs(output_dir_real, exist_ok=True)

# 用 vol_df 當來源
df_out = vol_df.copy()
df_out.index = pd.to_datetime(df_out.index, errors='coerce')
df_out = df_out.sort_index()

df_out = df_out.loc[
    (df_out.index >= EXPORT_START) & (df_out.index <= EXPORT_END)
].copy()

df_out = df_out.reset_index().rename(columns={'index': 'DATE'})

# 真實報酬
df_out['ret_true'] = pd.to_numeric(df_out[ret_col], errors='coerce')

# mu = 0
df_out['mu'] = 0.0

# 原始 shape + clipped shape
df_out['shape'] = pd.to_numeric(df_out['shape'], errors='coerce')
df_out['shape_clipped'] = df_out['shape'].clip(NU_CLIP[0], NU_CLIP[1])

# 價格
df_out['price_true'] = pd.to_numeric(df_out['ES1_CLOSE'], errors='coerce')
df_out['price_base_aligned'] = df_out['price_true'].shift(1)

# 這裡要把後面真的會用到的欄位全部保留下來
need_cols = [
    'DATE',
    'ret_true',
    'mu',
    'shape',
    'shape_clipped',
    'price_true',
    'price_base_aligned'
] + vol_cols

df_out = df_out[need_cols].copy()

# =========================================================
# 3. 主迴圈：每個波動欄位各自做 VaR/ES
# =========================================================
results_dict = {}
summary_rows = []

for vol_col in vol_cols:
    method_df = df_out[[
        'DATE',
        'ret_true',
        'price_true',
        'price_base_aligned',
        'shape',
        'shape_clipped',
        'mu',
        vol_col
    ]].copy()

    method_df['sigma_used'] = pd.to_numeric(method_df[vol_col], errors='coerce').clip(lower=0)
    method_df['var_used'] = method_df['sigma_used'] ** 2

    res = compute_var_es_auto(
        mu=method_df['mu'],
        sigma=method_df['sigma_used'],
        alpha=alpha,
        nu=method_df['shape_clipped'],
        price_base=method_df['price_base_aligned'],
        standardized_t=STD_T,
        nu_clip=NU_CLIP
    )

    method_df['VaR_ret_95'] = res['VaR_ret'].values
    method_df['ES_ret_95'] = res['ES_ret'].values

    if 'VaR_price' in res.columns:
        method_df['VaR_price_95'] = res['VaR_price'].values
    if 'ES_price' in res.columns:
        method_df['ES_price_95'] = res['ES_price'].values

    method_df['viol_95'] = (method_df['ret_true'] < method_df['VaR_ret_95']).astype(int)

    valid_df = method_df.dropna(
        subset=['DATE', 'ret_true', 'sigma_used', 'VaR_ret_95', 'shape_clipped']
    ).copy()

    if len(valid_df) == 0:
        print(f"[WARN] {vol_col} 無有效樣本，跳過。")
        continue

    lr_uc, p_uc, vio_rate = kupiec_uc_test(valid_df['viol_95'], p=alpha)
    lr_ind, p_ind, n00, n01, n10, n11 = christoffersen_cc_test(valid_df['viol_95'])

    lr_cc = lr_uc + lr_ind
    p_cc = 1 - chi2.cdf(lr_cc, df=2)

    results_dict[vol_col] = method_df.copy()

    daily_path = os.path.join(output_dir_real, f"{vol_col}_var_es_daily.csv")
    method_df.to_csv(daily_path, index=False, encoding='utf-8-sig')

    summary_rows.append({
        'method': vol_col,
        'n': len(valid_df),
        'viol_count_95': int(valid_df['viol_95'].sum()),
        'viol_rate_95': float(valid_df['viol_95'].mean()),
        'mean_var_used': float(valid_df['var_used'].mean()),
        'mean_sigma_used': float(valid_df['sigma_used'].mean()),
        'mean_VaR_ret_95': float(valid_df['VaR_ret_95'].mean()),
        'mean_ES_ret_95': float(valid_df['ES_ret_95'].mean()),
        'kupiec_lr_uc': float(lr_uc),
        'kupiec_pvalue': float(p_uc),
        'cc_lr_ind': float(lr_ind),
        'cc_pvalue_ind': float(p_ind),
        'cc_lr_cc': float(lr_cc),
        'cc_pvalue_cc': float(p_cc),
        'n00': int(n00),
        'n01': int(n01),
        'n10': int(n10),
        'n11': int(n11)
    })

    plt.figure(figsize=(12, 6))

    plt.plot(
        valid_df['DATE'],
        valid_df['ret_true'],
        label='True Return',
        alpha=0.7,
        linewidth=0.8
    )

    plt.plot(
        valid_df['DATE'],
        valid_df['VaR_ret_95'],
        label='VaR 95%',
        linewidth=0.8,
        alpha=0.9
    )

    viol = valid_df['viol_95'] == 1
    plt.scatter(
        valid_df.loc[viol, 'DATE'],
        valid_df.loc[viol, 'ret_true'],
        color='red',
        s=8,
        alpha=0.9,
        label='Violation'
    )

    plt.axhline(0, linewidth=0.8, linestyle='--')
    plt.xlabel("Time")
    plt.ylabel("Log Return")
    plt.title(f"{vol_col} - Return vs VaR(95%)")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()

    fig_path = os.path.join(output_dir_real, f"{vol_col}_VaR95_plot.png")
    plt.savefig(fig_path, dpi=300, bbox_inches='tight')
    plt.close()

# =========================================================
# 4. 摘要 DataFrame
# =========================================================
summary_df = pd.DataFrame(summary_rows)
summary_df = summary_df.sort_values('method').reset_index(drop=True)

summary_path = os.path.join(output_dir_real, "all_methods_var95_summary.csv")
summary_df.to_csv(summary_path, index=False, encoding='utf-8-sig')

print("\n[summary_df]")
print(summary_df)

print("\nresults_dict keys:")
print(list(results_dict.keys()))

print(f"\n全部結果已存到：{output_dir_real}")


[summary_df]
                method     n  viol_count_95  viol_rate_95  mean_var_used  \
0             ewma_vol  5030            285      0.056660       0.000150   
1            garch_vol  5030            299      0.059443       0.000158   
2            gk_vol_10  5030            293      0.058250       0.000137   
3            gk_vol_20  5030            303      0.060239       0.000137   
4             gk_vol_5  5030            284      0.056461       0.000137   
5         gk_vol_daily  5030            222      0.044135       0.000137   
6            rs_vol_10  5030            300      0.059642       0.000136   
7            rs_vol_20  5030            310      0.061630       0.000136   
8             rs_vol_5  5030            302      0.060040       0.000136   
9         rs_vol_daily  5030            310      0.061630       0.000136   
10          sma_vol_10  5030            366      0.072763       0.000139   
11  sma_vol_10_predict  5030            417      0.082903       0.000139  